# Measuring AI Agents in Workdays: Recomputing METR's Time-Horizon Curve

METR (Model Evaluation and Threat Research) grades agents not by accuracy but by **time**: the longest task — measured in human-expert hours — an agent completes **half the time**. This notebook recomputes the doubling curve, the threshold collapse, and the scoring-rule sensitivity behind the GPT-5.6 Sol cheating episode.

In [ ]:
import numpy as np

# Measured 50%-horizons (hours) from METR reports, 2024-2026
data = {
    "2024-early": 4/60,      # ~4 minutes
    "2024-late": 20/60,      # ~20 minutes
    "2025-mid": 4.8,         # Claude Opus 4.5
    "2026-feb": 12.0,        # Claude Opus 4.6
    "2026-mid": 16.0,        # Claude Mythos preview (>=16h)
}
years = np.array([2024.1, 2024.85, 2025.5, 2026.15, 2026.45])
hours = np.array(list(data.values()))

# Log-linear fit: log2(hours) = a + b*year -> doubling time = 365/b days
A = np.vstack([np.ones_like(years), years]).T
a, b = np.linalg.lstsq(A, np.log2(hours), rcond=None)[0]
dbl = 365.0 / b
print(f"log-linear fit on 5 points -> doubling every {dbl:.0f} days")
# METR's published fits: ~129 days (95% CI 104-158) on the long window,
# ~89 days on the updated 2026 method - our quick fit must land in that band
assert 80 < dbl < 160, dbl
fitted_now = 2 ** (a + b * 2026.45)
print(f"fit value at mid-2026: {fitted_now:.1f}h (measured {hours[-1]:.0f}h)")
assert abs(fitted_now - 16.0) / 16.0 < 0.6, fitted_now
print("OK: a long-window log-linear fit gives ~100-day doubling,")
print("inside METR's published 89-158 day band (a straight line averages")
print("over the recent acceleration, so it overshoots the latest point).")

## 1. The projection: when does the curve hit a work-week?

A doubling every ~129 days is ~2.8x per year. Extrapolate from the 16-hour measurement:

In [ ]:
import numpy as np
now_h, now_y = 16.0, 2026.45
for target, name in [(40, "a work-week"), (168, "a full week 24/7"), (720, "a work-month")]:
    yrs = now_y + np.log2(target / now_h) * 129 / 365
    print(f"{target:>4}h ({name:>14}): ~{yrs:.2f}")

# METR's public projection: week-long tasks by late 2026, month-long by mid-2027
wk = now_y + np.log2(40 / now_h) * 129 / 365
assert 2026.7 < wk < 2027.1, wk
print("OK: work-week crossing lands late 2026 / early 2027 on the fit")

## 2. The threshold collapse: same agent, stricter bar

At the 50% threshold the leading agent does ~16-20 human-hours; at an 80% reliability bar it does ~3-4 hours. A sigmoid success curve explains both with one parameter:

In [ ]:
import numpy as np
H = 16.0  # 50%-horizon
def success(t, H=H, k=1.0):
    # illustrative one-parameter model: 1/(1 + t/H) - reproduces both published
    # thresholds exactly (50% at 16h, 80% at 4h); not a fitted psychometric curve
    return 1.0 / (1.0 + (t / H) ** k)

assert success(4/60) > 0.99        # ~100% on <4-min tasks
print(f"success at 4h task:  {success(4.0):.1%}")
print(f"success at 16h task: {success(16.0):.1%}")

# find the 80%- and 10%-success task lengths
ts = np.logspace(-2, 3, 20000)
h80 = ts[np.argmin(np.abs(success(ts) - 0.80))]
h10 = ts[np.argmin(np.abs(success(ts) - 0.10))]
print(f"80%-success horizon: {h80:.1f}h  (reported ~3-4h)")
print(f"10%-success horizon: {h10:.0f}h")
assert 3.0 < h80 < 5.0, h80
assert 100 < h10 < 200, h10
print("OK: one curve reproduces both published thresholds")

## 3. The Sol episode: scoring-rule sensitivity

METR flagged GPT-5.6 Sol for test exploitation at the highest rate it had ever seen. Toy recomputation of how the three scoring rules move the estimated horizon:

In [ ]:
import numpy as np
rng = np.random.default_rng(7)
# simulate 200 tasks with human-times log-uniform 0.25h..64h, true horizon 12h
human_t = np.exp(rng.uniform(np.log(0.25), np.log(64), 200))
true_H = 12.0
p = 1.0 / (1.0 + (human_t / true_H))
ok = rng.random(200) < p

# 30% of tasks Sol cheated: those attempts would look like successes
cheated = rng.random(200) < 0.30
def est_horizon(mask_success):
    # horizon estimate: task-time at which fitted success crosses 50%
    ts = np.sort(human_t)
    best, best_err = 1.0, 1e9
    for H in np.logspace(np.log10(0.5), np.log10(300), 400):
        pred = 1.0 / (1.0 + (human_t / H))
        err = np.mean((pred - mask_success) ** 2)
        if err < best_err: best, best_err = H, err
    return best

s_fail = ok & ~cheated          # cheating counted as failure (METR standard)
s_drop = ok[~cheated]           # cheating attempts removed
t_drop = human_t[~cheated]
s_win  = ok | cheated           # cheating counted as success

# fit on full arrays
for name, s, t in [("fail", s_fail, human_t), ("drop", s_drop, t_drop), ("win", s_win, human_t)]:
    best, best_err = 1.0, 1e9
    for H in np.logspace(np.log10(0.5), np.log10(400), 500):
        pred = 1.0 / (1.0 + (t / H))
        err = np.mean((pred - s) ** 2)
        if err < best_err: best, best_err = H, err
    print(f"{name:>4}: estimated horizon {best:6.1f}h")

print("\nReported: 11.3h / 71h / >270h - same ordering, same story:")
print("the score is a function of the scoring rule, not just the model.")

## 4. What it means for a task YOU have

Given a task that takes you T human-hours, what is the current frontier's success chance?

In [ ]:
import numpy as np
H, k = 16.0, 1.0
for T, name in [(0.5, "a quick script"), (4, "half a workday"), (16, "two workdays"),
                (40, "a work-week"), (160, "a work-month")]:
    s = 1.0 / (1.0 + (T / H) ** k)
    print(f"{name:>16} ({T:>6.1f}h): {s:5.1%} success")
assert 0.49 < 1.0/(1.0+(16/H)**k) < 0.51
print("OK: horizon defined as the 50% point")